# 5. Recommendation Function

### 1. Import Libraries & Load Dataset 

In [44]:
import pandas as pd
import streamlit as st

In [45]:
df = pd.read_csv("../data/processed/mvp_herbal_dataset_v3.csv")
df.head()

,Common Name,Scientific Name,Medicinal Properties,Known Hazards,Summary,user_need_tags,effect_tags,preparation_type,warning_level,recommendation_reason
0,"Musk Mallow,Musk Okra",Abelmoschus moschatus - Medik.,Antihalitosis; Antispasmodic; Aphrodisiac; App...,None known,NaN,"anxiety,digestion,relaxation,skin,sleep,stress","antihalitosis, antispasmodic, aphrodisiac, app...","oil,topical",low,
1,"Grand Fir, Giant Fir, Lowland White Fir",Abies grandis - (Douglas. ex D.Don.)Lindl.,Antirheumatic; Laxative; Ophthalmic; Skin; Sto...,None known,"Form: Columnar, Upright or erect.","digestion,energy,focus,pain","antirheumatic, laxative, ophthalmic, skin, sto...",topical,low,
2,Himalayan Fir,Abies spectabilis - (D.Don.)Spach.,Antiperiodic; Astringent; Carminative; Expecto...,None known,NaN,"digestion,energy,focus,respiratory_support","antiperiodic, astringent, carminative, expecto...",unknown,low,
3,"China Jute, Velvetleaf, Butterprint Buttonweed...",Abutilon theophrasti - Medik.,Astringent; Demulcent; Diuretic; Dysentery; Em...,None known,Form: Upright or erect.,"digestion,edema,skin","astringent, demulcent, diuretic, dysentery, em...",topical,low,
4,"Sweet Acacia, Perfume Acacia, Huisache",Acacia farnesiana - (L.)Willd.,Astringent; Demulcent; Dysentery; Poultice; St...,"The seeds, containing an unnamed alkaloid, are...",Bloom Color: Yellow. Main Bloom Time: Early su...,"digestion,skin","astringent, demulcent, dysentery, poultice, st...",topical,medium,


### 2. Prepare Tags for Matching

In [46]:
def split_tags(tag_string):
    if pd.isna(tag_string) or tag_string == "":
        return []
    
    return [tag.strip() for tag in tag_string.split(",")]

In [47]:
split_tags("digestion,pain,skin")

['digestion', 'pain', 'skin']

### 3. Calculate Matching Score

In [48]:
# score = matching tags / user input tags

In [49]:
def calculate_match_score(user_tags, plant_tags):
    user_tags = set(user_tags)
    plant_tags = set(plant_tags)

    if len(user_tags) == 0:
        return 0
    
    matches = user_tags.intersection(plant_tags)

    score = len(matches) / len(user_tags)

    return score

In [50]:
calculate_match_score(
    ["digestion", "pain"],
    ["digestion", "skin", "pain"]
)

1.0

In [51]:
calculate_match_score(
    ["anxiety", "pain"],
    ["anxiety", "sleep", "stress"]
)

0.5

In [52]:
calculate_match_score(
    ["anxiety", "pain"],
    ["digestion", "sleep", "stress"]
)

0.0

In [53]:
def format_score(score):
    if score == 1.0:
        return "Very strong match"
    elif score >= 0.6:
        return "Good match"
    elif score > 0:
        return "Partial match"
    else:
        return "No match"

### 4. Build Recommendation Function

In [54]:
def recommend_plants(user_tags, dataset, top_n=5):
    recommendations = dataset.copy()

    # Convert plant tags into lists
    recommendations["plant_tag_list"] = recommendations["user_need_tags"].apply(split_tags)

    # Calculate matching score
    recommendations["recommendation_score"] = recommendations["plant_tag_list"].apply(
        lambda plant_tags: calculate_match_score(user_tags, plant_tags)
    )

    # Safety filter: exclude high-risk plants
    recommendations = recommendations[
        recommendations["warning_level"] != "high"
    ]

    # Keep only plants with score above 0
    recommendations = recommendations[
        recommendations["recommendation_score"] > 0
    ]

    # Sort by score
    recommendations = recommendations.sort_values(
        by="recommendation_score",
        ascending=False
    )

    return recommendations[
        [
            "Common Name",
            "Scientific Name",
            "user_need_tags",
            "effect_tags",
            "preparation_type",
            "warning_level",
            "recommendation_score"
        ]
    ].head(top_n)

### 5. Test Recommendation Function

In [55]:
# first test
recommend_plants(
    user_tags=["digestion", "pain"],
    dataset=df,
    top_n=5
)

,Common Name,Scientific Name,user_need_tags,effect_tags,preparation_type,warning_level,recommendation_score
99,"Chamomile, Roman chamomile",Chamaemelum nobile - (L.)All.,"anxiety,circulation,digestion,energy,focus,hea...","anodyne, antianxiety, antiinflammatory, antisp...",oil,medium,1.0
145,"Agar Wood, Eaglewood, Indian Aloewood, Aloeswood",Aquilaria malaccensis - Lam.,"anxiety,digestion,energy,focus,pain,relaxation...","antiasthmatic, antidiarrhoeal, antirheumatic, ...",not recommended due to safety concerns,medium,1.0
29,Speckled Alder,Alnus rugosa - (Du Roi.)Spreng.,"digestion,energy,focus,pain","alterative, anodyne, astringent, cathartic, em...",unknown,low,1.0
78,Dwarf Birch,Betula nana - L.,"anxiety,digestion,pain,relaxation,sleep,stress","antirheumatic, antiseborrheic, astringent, lit...",cream_or_onitment,low,1.0
31,"Mountain Alder, Thinleaf alder",Alnus tenuifolia - Nutt.,"digestion,energy,focus,pain","anodyne, astringent, emetic, febrifuge, haemos...",capsule_or_powder,medium,1.0


In [56]:
# second test
recommend_plants(
    user_tags=["stress", "sleep"],
    dataset=df,
    top_n=5
)

,Common Name,Scientific Name,user_need_tags,effect_tags,preparation_type,warning_level,recommendation_score
0,"Musk Mallow,Musk Okra",Abelmoschus moschatus - Medik.,"anxiety,digestion,relaxation,skin,sleep,stress","antihalitosis, antispasmodic, aphrodisiac, app...","oil,topical",low,1.0
127,Lemon Balm,Melissa officinalis - L.,"anxiety,circulation,digestion,energy,focus,imm...","antianxiety, antibacterial, antidepressant, an...",not recommended due to safety concerns,medium,1.0
166,"Valerian, Garden valerian",Valeriana officinalis - L.,"anxiety,digestion,edema,energy,focus,relaxatio...","anticonvulsant, antispasmodic, carminative, di...",not recommended due to safety concerns,medium,1.0
162,Wild Ginger,Asarum sieboldii - Miq.,"anxiety,circulation,edema,heart_support,immune...","anaesthetic, analgesic, antibacterial, antipyr...",unknown,medium,1.0
149,"Maypops - Passion Flower, Purple passionflower...",Passiflora incarnata - L.,"anxiety,circulation,heart_support,relaxation,s...","antidepressant, antispasmodic, astringent, dia...",unknown,medium,1.0


In [57]:
# third test
recommend_plants(
    user_tags=["skin", "inflammation"],
    dataset=df,
    top_n=50
)

,Common Name,Scientific Name,user_need_tags,effect_tags,preparation_type,warning_level,recommendation_score
0,"Musk Mallow,Musk Okra",Abelmoschus moschatus - Medik.,"anxiety,digestion,relaxation,skin,sleep,stress","antihalitosis, antispasmodic, aphrodisiac, app...","oil,topical",low,0.5
3,"China Jute, Velvetleaf, Butterprint Buttonweed...",Abutilon theophrasti - Medik.,"digestion,edema,skin","astringent, demulcent, diuretic, dysentery, em...",topical,low,0.5
196,"Mountain Dandelion, Pale agoseris, False agoseris",Agoseris glauca - (Pursh.)Raf.,"digestion,skin","laxative, poultice, warts",topical,low,0.5
185,Lemon Scented Thyme,Micromeria biflora - (Buch.-Ham. ex D.Don.)Benth.,"immune_support,pain,skin","antiseptic, odontalgic, vulnerary",topical,low,0.5
168,"White Sage, Louisiana Sage, Prairie Sage, West...",Artemisia ludoviciana - Nutt.,skin,"astringent, deodorant, eczema, poultice, skin",topical,medium,0.5
158,"Lungwort, Common lungwort, Jerusalem Sage, Jer...",Pulmonaria officinalis - L.,"circulation,edema,respiratory_support,skin","astringent, demulcent, diaphoretic, diuretic, ...",topical,low,0.5
150,Garlic Mustard,Alliaria petiolata - (M.Bieb.)Cavara.&Grande.,"circulation,immune_support,respiratory_support...","antiasthmatic, antiscorbutic, antiseptic, deob...",topical,low,0.5
136,Wood Sage,Teucrium scorodonia - L.,"circulation,digestion,edema,energy,focus,skin","alterative, appetizer, astringent, carminative...",topical,low,0.5
135,"Marsh Woundwort, Marsh hedgenettle",Stachys palustris - L.,"anxiety,energy,focus,immune_support,relaxation...","antiseptic, antispasmodic, emetic, emmenagogue...",not recommended due to safety concerns,medium,0.5
134,"Wood Betony, Common hedgenettle, Betony, Wound...",Stachys officinalis - (L.)Trevis.,"anxiety,digestion,edema,energy,focus,immune_su...","anthelmintic, antiseptic, astringent, carminat...",topical,low,0.5


#

# 6. Improve Recommendation Output

### 1. Get matched tags

In [58]:
def get_matched_tags(user_tags, plant_tags):
    user_tags = set(user_tags)
    plant_tags = set(plant_tags)

    matched_tags = user_tags.intersection(plant_tags)

    return list(matched_tags)

### 2. Generate Recommendation Reason

In [59]:
def generate_recommendation_reason(matched_tags):
    if len(matched_tags) == 0:
        return "No strong match found."
    
    matched_text = ", ".join(matched_tags)

    return f"Recommended because this plant matches your needs: {matched_text}."

### 3. Generate safety note

In [60]:
def generate_safety_note(warning_level):
    if warning_level == "low":
        return "Low warning level based on available hazard information."
    elif warning_level == "medium":
        return "Medium warning level. Please review known hazards before use."
    elif warning_level == "high":
        return "High warning level. This plant is not recommended."
    else:
        return "Warning level unknown."

### 4. Build imporved function

In [61]:
def recommend_plants_v2(user_tags, dataset, top_n=5):
    recommendations = dataset.copy()

    # Convert plant tags into lists
    recommendations["plant_tag_list"] = recommendations["user_need_tags"].apply(split_tags)

    # Get matched tags
    recommendations["matched_tags"] = recommendations["plant_tag_list"].apply(
        lambda plant_tags: get_matched_tags(user_tags, plant_tags)
    )

    # Calculate recommendation score
    recommendations["recommendation_score"] = recommendations["matched_tags"].apply(
        lambda matched_tags: len(matched_tags) / len(user_tags) if len(user_tags) > 0 else 0
    )

    # Safety filter: exclude high-risk plants
    recommendations = recommendations[
        recommendations["warning_level"] != "high"
    ]

    # Keep only plants with score above 0
    recommendations = recommendations[
        recommendations["recommendation_score"] > 0
    ]

    # Generate explanation
    recommendations["recommendation_reason"] = recommendations["matched_tags"].apply(
        generate_recommendation_reason
    )

    # Generate safety note
    recommendations["safety_note"] = recommendations["warning_level"].apply(
        generate_safety_note
    )

    # Sort by score
    recommendations = recommendations.sort_values(
        by="recommendation_score",
        ascending=False
    )

    return recommendations[
        [
            "Common Name",
            "Scientific Name",
            "matched_tags",
            "user_need_tags",
            "effect_tags",
            "preparation_type",
            "warning_level",
            "recommendation_score",
            "recommendation_reason",
            "safety_note"
        ]
    ].head(top_n)

### 5. Test improved recommendation function

In [62]:
# first test
recommend_plants_v2(
    user_tags=["skin", "infammation"],
    dataset=df,
    top_n=5
)

,Common Name,Scientific Name,matched_tags,user_need_tags,effect_tags,preparation_type,warning_level,recommendation_score,recommendation_reason,safety_note
0,"Musk Mallow,Musk Okra",Abelmoschus moschatus - Medik.,[skin],"anxiety,digestion,relaxation,skin,sleep,stress","antihalitosis, antispasmodic, aphrodisiac, app...","oil,topical",low,0.5,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
3,"China Jute, Velvetleaf, Butterprint Buttonweed...",Abutilon theophrasti - Medik.,[skin],"digestion,edema,skin","astringent, demulcent, diuretic, dysentery, em...",topical,low,0.5,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
196,"Mountain Dandelion, Pale agoseris, False agoseris",Agoseris glauca - (Pursh.)Raf.,[skin],"digestion,skin","laxative, poultice, warts",topical,low,0.5,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
185,Lemon Scented Thyme,Micromeria biflora - (Buch.-Ham. ex D.Don.)Benth.,[skin],"immune_support,pain,skin","antiseptic, odontalgic, vulnerary",topical,low,0.5,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
168,"White Sage, Louisiana Sage, Prairie Sage, West...",Artemisia ludoviciana - Nutt.,[skin],skin,"astringent, deodorant, eczema, poultice, skin",topical,medium,0.5,Recommended because this plant matches your ne...,Medium warning level. Please review known haza...


In [63]:
# second test
recommend_plants_v2(
    user_tags=["stress", "sleep"],
    dataset=df,
    top_n=5
)

,Common Name,Scientific Name,matched_tags,user_need_tags,effect_tags,preparation_type,warning_level,recommendation_score,recommendation_reason,safety_note
0,"Musk Mallow,Musk Okra",Abelmoschus moschatus - Medik.,"[sleep, stress]","anxiety,digestion,relaxation,skin,sleep,stress","antihalitosis, antispasmodic, aphrodisiac, app...","oil,topical",low,1.0,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
127,Lemon Balm,Melissa officinalis - L.,"[sleep, stress]","anxiety,circulation,digestion,energy,focus,imm...","antianxiety, antibacterial, antidepressant, an...",not recommended due to safety concerns,medium,1.0,Recommended because this plant matches your ne...,Medium warning level. Please review known haza...
166,"Valerian, Garden valerian",Valeriana officinalis - L.,"[sleep, stress]","anxiety,digestion,edema,energy,focus,relaxatio...","anticonvulsant, antispasmodic, carminative, di...",not recommended due to safety concerns,medium,1.0,Recommended because this plant matches your ne...,Medium warning level. Please review known haza...
162,Wild Ginger,Asarum sieboldii - Miq.,"[sleep, stress]","anxiety,circulation,edema,heart_support,immune...","anaesthetic, analgesic, antibacterial, antipyr...",unknown,medium,1.0,Recommended because this plant matches your ne...,Medium warning level. Please review known haza...
149,"Maypops - Passion Flower, Purple passionflower...",Passiflora incarnata - L.,"[sleep, stress]","anxiety,circulation,heart_support,relaxation,s...","antidepressant, antispasmodic, astringent, dia...",unknown,medium,1.0,Recommended because this plant matches your ne...,Medium warning level. Please review known haza...


In [64]:
recommend_plants_v2(
    user_tags=["digestion", "pain"],
    dataset=df,
    top_n=5
)

,Common Name,Scientific Name,matched_tags,user_need_tags,effect_tags,preparation_type,warning_level,recommendation_score,recommendation_reason,safety_note
99,"Chamomile, Roman chamomile",Chamaemelum nobile - (L.)All.,"[digestion, pain]","anxiety,circulation,digestion,energy,focus,hea...","anodyne, antianxiety, antiinflammatory, antisp...",oil,medium,1.0,Recommended because this plant matches your ne...,Medium warning level. Please review known haza...
145,"Agar Wood, Eaglewood, Indian Aloewood, Aloeswood",Aquilaria malaccensis - Lam.,"[digestion, pain]","anxiety,digestion,energy,focus,pain,relaxation...","antiasthmatic, antidiarrhoeal, antirheumatic, ...",not recommended due to safety concerns,medium,1.0,Recommended because this plant matches your ne...,Medium warning level. Please review known haza...
29,Speckled Alder,Alnus rugosa - (Du Roi.)Spreng.,"[digestion, pain]","digestion,energy,focus,pain","alterative, anodyne, astringent, cathartic, em...",unknown,low,1.0,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
78,Dwarf Birch,Betula nana - L.,"[digestion, pain]","anxiety,digestion,pain,relaxation,sleep,stress","antirheumatic, antiseborrheic, astringent, lit...",cream_or_onitment,low,1.0,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
31,"Mountain Alder, Thinleaf alder",Alnus tenuifolia - Nutt.,"[digestion, pain]","digestion,energy,focus,pain","anodyne, astringent, emetic, febrifuge, haemos...",capsule_or_powder,medium,1.0,Recommended because this plant matches your ne...,Medium warning level. Please review known haza...


#

# 7. Convert User Text into Tags

### 1. Converting text to tags

In [65]:
user_need_keywords = {
    "digestion": [
        "stomach", "belly", "digest", "digestion", "nausea", "bloating", "gas", "cramps", "constipation", "diarrhea"
    ],
    "pain": [
        "pain", "ache", "headache", "cramps", "sore", "hurt"
    ],
    "cramps": [
        "cramp", "cramps", "period cramps", "menstrual cramps", "period pain", "mensturation pain", "menstrual pain"
    ],
    "menstrual support": [
        "period", "menstruation", "menstrual", "pms"
    ],
    "stress": [
        "stress", "stressed", "overwhelmed", "pressure", "tense"
    ],
    "sleep": [
        "sleep", "insomnia", "can't sleep", "cannot sleep", "sleepless", "tired at night"
    ],
    "anxiety": [
        "anxiety", "anxious", "worry", "worried", "nervous"
    ],
    "relaxation": [
        "relax", "relaxation", "calm", "claming", "restless"
    ],
    "skin": [
        "skin", "rash", "wound", "burn", "eczema", "itchy", "irritation"
    ],
    "inflammation": [
        "infammation", "inflamed", "swelling", "anti inflammatory"
    ],
    "immune_support": [
        "immune", "immunity", "cold", "flu", "infection"
    ],
    "respiratory_support": [
        "cough", "breathing", "respiratory", "asthma", "bronchitis"
    ],
    "energy": [
        "energy", "tired", "fatigue", "weak", "exhausted"
    ],
    "focus": [
        "focus", "concentration", "memory", "mental", "mental clarity"
    ],
    "edema": [
        "edema", "water retention", "swelling", "fluid retention"
    ]
}

### 2. Build the function

In [66]:
def extract_tags_from_user_text(user_text):
    user_text = user_text.lower()
    extracted_tags = []

    for tag, keywords in user_need_keywords.items():
        for keyword in keywords:
            if keyword in user_text:
                extracted_tags.append(tag)
                break
    
    return sorted(set(extracted_tags))

### 3. Test the function

In [67]:
# first test
extract_tags_from_user_text("I have stomach pain and feel stressed.")

['digestion', 'pain', 'stress']

In [68]:
# second test
extract_tags_from_user_text("I cannot sleep and I feel very anxious.")

['anxiety', 'sleep']

### 4. Connecting with Recommendation function

In [69]:
user_text = "I have stomach pain and feel stressed."
detected_tags = extract_tags_from_user_text(user_text)

recommend_plants_v2(
    user_tags=detected_tags,
    dataset=df,
    top_n=5
)

,Common Name,Scientific Name,matched_tags,user_need_tags,effect_tags,preparation_type,warning_level,recommendation_score,recommendation_reason,safety_note
47,Tian Nan Xing,Arisaema consanguineum - Schott.,"[digestion, stress, pain]","anxiety,digestion,immune_support,inflammatory,...","anodyne, antibacterial, anticoagulant, antifun...",food,medium,1.0,Recommended because this plant matches your ne...,Medium warning level. Please review known haza...
99,"Chamomile, Roman chamomile",Chamaemelum nobile - (L.)All.,"[digestion, stress, pain]","anxiety,circulation,digestion,energy,focus,hea...","anodyne, antianxiety, antiinflammatory, antisp...",oil,medium,1.0,Recommended because this plant matches your ne...,Medium warning level. Please review known haza...
78,Dwarf Birch,Betula nana - L.,"[digestion, stress, pain]","anxiety,digestion,pain,relaxation,sleep,stress","antirheumatic, antiseborrheic, astringent, lit...",cream_or_onitment,low,1.0,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
138,Wild Thyme,Thymus serpyllum - L.,"[digestion, stress, pain]","anxiety,circulation,digestion,energy,focus,imm...","anthelmintic, antirheumatic, antiseptic, antis...",food,low,1.0,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
145,"Agar Wood, Eaglewood, Indian Aloewood, Aloeswood",Aquilaria malaccensis - Lam.,"[digestion, stress, pain]","anxiety,digestion,energy,focus,pain,relaxation...","antiasthmatic, antidiarrhoeal, antirheumatic, ...",not recommended due to safety concerns,medium,1.0,Recommended because this plant matches your ne...,Medium warning level. Please review known haza...


In [70]:
user_text = "My skin is irritated and inflamed."
detected_tags = extract_tags_from_user_text(user_text)

recommend_plants_v2(
    user_tags=detected_tags,
    dataset=df,
    top_n=5
)

,Common Name,Scientific Name,matched_tags,user_need_tags,effect_tags,preparation_type,warning_level,recommendation_score,recommendation_reason,safety_note
0,"Musk Mallow,Musk Okra",Abelmoschus moschatus - Medik.,[skin],"anxiety,digestion,relaxation,skin,sleep,stress","antihalitosis, antispasmodic, aphrodisiac, app...","oil,topical",low,0.5,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
3,"China Jute, Velvetleaf, Butterprint Buttonweed...",Abutilon theophrasti - Medik.,[skin],"digestion,edema,skin","astringent, demulcent, diuretic, dysentery, em...",topical,low,0.5,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
196,"Mountain Dandelion, Pale agoseris, False agoseris",Agoseris glauca - (Pursh.)Raf.,[skin],"digestion,skin","laxative, poultice, warts",topical,low,0.5,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
185,Lemon Scented Thyme,Micromeria biflora - (Buch.-Ham. ex D.Don.)Benth.,[skin],"immune_support,pain,skin","antiseptic, odontalgic, vulnerary",topical,low,0.5,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
168,"White Sage, Louisiana Sage, Prairie Sage, West...",Artemisia ludoviciana - Nutt.,[skin],skin,"astringent, deodorant, eczema, poultice, skin",topical,medium,0.5,Recommended because this plant matches your ne...,Medium warning level. Please review known haza...


In [71]:
user_text = "I feel tired and have no energy."
detected_tags = extract_tags_from_user_text(user_text)

recommend_plants_v2(
    user_tags=detected_tags,
    dataset=df,
    top_n=5
)

,Common Name,Scientific Name,matched_tags,user_need_tags,effect_tags,preparation_type,warning_level,recommendation_score,recommendation_reason,safety_note
1,"Grand Fir, Giant Fir, Lowland White Fir",Abies grandis - (Douglas. ex D.Don.)Lindl.,[energy],"digestion,energy,focus,pain","antirheumatic, laxative, ophthalmic, skin, sto...",topical,low,1.0,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
120,"Mondia, White's Ginger",Mondia whitei - (Hook.f.) Skeels,[energy],"digestion,edema,energy,focus,inflammatory,resp...","antidepressant, antiemetic, antiinflammatory, ...","capsule_or_powder,food",medium,1.0,Recommended because this plant matches your ne...,Medium warning level. Please review known haza...
130,Spanish Sage,Salvia lavandulifolia - Vahl.,[energy],"digestion,energy,focus,immune_support,respirat...","alterative, antiseptic, astringent, depurative...",unknown,low,1.0,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
129,"Greek Sage, Greek oregano",Salvia fruticosa - Mill.,[energy],"circulation,digestion,energy,focus,heart_suppo...","antihydrotic, antiseptic, antispasmodic, astri...","food,tea",low,1.0,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
128,"Wax Myrtle - Bayberry Wild Cinnamon, Southern ...",Myrica cerifera - L.,[energy],"energy,focus,immune_support","antibacterial, astringent, dysentery, emetic, ...",not recommended due to safety concerns,medium,1.0,Recommended because this plant matches your ne...,Medium warning level. Please review known haza...


#

# 8. Handling Unclear User Input

### 1. Building the function for unmatched tags

In [72]:
def recommend_from_user_text(user_text, dataset, top_n=5):
    detected_tags = extract_tags_from_user_text(user_text)

    if len(detected_tags) == 0:
        return "No matching tags found. Please try words like sleep, stress, pain, skin, inflammation, etc."
    
    return recommend_plants_v2(
        user_tags=detected_tags,
        dataset=dataset,
        top_n=top_n
    )

### 2. Test the function

In [73]:
# Test 1: Right Input
recommend_from_user_text(
    "I have stomach pain and feel stressed.",
    dataset=df,
    top_n=5
)

,Common Name,Scientific Name,matched_tags,user_need_tags,effect_tags,preparation_type,warning_level,recommendation_score,recommendation_reason,safety_note
47,Tian Nan Xing,Arisaema consanguineum - Schott.,"[digestion, stress, pain]","anxiety,digestion,immune_support,inflammatory,...","anodyne, antibacterial, anticoagulant, antifun...",food,medium,1.0,Recommended because this plant matches your ne...,Medium warning level. Please review known haza...
99,"Chamomile, Roman chamomile",Chamaemelum nobile - (L.)All.,"[digestion, stress, pain]","anxiety,circulation,digestion,energy,focus,hea...","anodyne, antianxiety, antiinflammatory, antisp...",oil,medium,1.0,Recommended because this plant matches your ne...,Medium warning level. Please review known haza...
78,Dwarf Birch,Betula nana - L.,"[digestion, stress, pain]","anxiety,digestion,pain,relaxation,sleep,stress","antirheumatic, antiseborrheic, astringent, lit...",cream_or_onitment,low,1.0,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
138,Wild Thyme,Thymus serpyllum - L.,"[digestion, stress, pain]","anxiety,circulation,digestion,energy,focus,imm...","anthelmintic, antirheumatic, antiseptic, antis...",food,low,1.0,Recommended because this plant matches your ne...,Low warning level based on available hazard in...
145,"Agar Wood, Eaglewood, Indian Aloewood, Aloeswood",Aquilaria malaccensis - Lam.,"[digestion, stress, pain]","anxiety,digestion,energy,focus,pain,relaxation...","antiasthmatic, antidiarrhoeal, antirheumatic, ...",not recommended due to safety concerns,medium,1.0,Recommended because this plant matches your ne...,Medium warning level. Please review known haza...


In [74]:
# Test 2: Unclear Input
recommend_from_user_text(
    "I feel weird today.",
    dataset=df,
    top_n=5
)

'No matching tags found. Please try words like sleep, stress, pain, skin, inflammation, etc.'